In [1]:
import ast
import uuid

from dotenv import load_dotenv
load_dotenv()

from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

from qdrant_client import QdrantClient, models
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage


/home/chirag/Documents/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
COLLECTION_NAME = "ms_marco_hybrid"

client = QdrantClient(
    url="http://localhost:6333"
)

# Dense
dense_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Sparse BM25
sparse_model = SparseTextEmbedding(
    model_name="Qdrant/bm25"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2823.27it/s]


In [3]:
# ============================================================
# Hybrid Search: Dense + Sparse BM25 Retrieval
# ============================================================

def hybrid_search(
    query: str,
    limit: int = 5,
    candidate_limit: int = 20
):
    """
    Perform hybrid search using dense semantic embeddings
    and sparse BM25 embeddings.

    The search process consists of:
        1. Generate a dense embedding for the query.
        2. Generate a sparse BM25 embedding for the query.
        3. Retrieve candidates independently from both
           dense and sparse vector spaces.
        4. Combine the results using Reciprocal Rank Fusion (RRF).
        5. Return the final top-ranked results.

    Args:
        query (str):
            The search query.

        limit (int):
            Number of final results to return.

        candidate_limit (int):
            Number of candidate results retrieved from each
            search method before applying RRF fusion.

    Returns:
        List of Qdrant search results.
    """

    # ========================================================
    # 1. Generate Dense Query Embedding
    # ========================================================
    # Convert the query into a dense semantic vector using
    # the same SentenceTransformer model used during indexing.
    #
    # Normalization allows the dense vector to work correctly
    # with cosine similarity.
    # ========================================================

    dense_vector = dense_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()


    # ========================================================
    # 2. Generate Sparse BM25 Query Embedding
    # ========================================================
    # Convert the query into a sparse BM25 representation.
    #
    # BM25 provides lexical matching, which is particularly
    # useful when exact keywords or terms are important.
    # ========================================================

    sparse_vector = list(
        sparse_model.embed([query])
    )[0]


    # Convert the sparse embedding into Qdrant's
    # SparseVector format.
    sparse_query = models.SparseVector(
        indices=sparse_vector.indices.tolist(),
        values=sparse_vector.values.tolist()
    )


    # ========================================================
    # 3. Perform Hybrid Search in Qdrant
    # ========================================================
    # Qdrant performs two independent candidate searches:
    #
    #   - Dense search  -> semantic similarity
    #   - Sparse search -> BM25 lexical matching
    #
    # The candidate results are then combined using
    # Reciprocal Rank Fusion (RRF).
    # ========================================================

    results = client.query_points(
        collection_name=COLLECTION_NAME,

        # ----------------------------------------------------
        # Retrieve candidate results from both vector spaces
        # ----------------------------------------------------

        prefetch=[
            # Dense semantic candidates
            models.Prefetch(
                query=dense_vector,
                using="dense",
                limit=candidate_limit
            ),

            # Sparse BM25 candidates
            models.Prefetch(
                query=sparse_query,
                using="sparse",
                limit=candidate_limit
            )
        ],

        # ----------------------------------------------------
        # Combine dense and sparse rankings using RRF
        # ----------------------------------------------------

        query=models.FusionQuery(
            fusion=models.Fusion.RRF
        ),

        # Number of final results returned after fusion
        limit=limit,

        # Include stored metadata/payload in the results
        with_payload=True
    )


    # ========================================================
    # 4. Return Final Hybrid Results
    # ========================================================

    return results.points

In [4]:
# ============================================================
# Initialize Language Model
# ============================================================

# Initialize the OpenAI chat model used for the
# answer-generation step in the RAG pipeline.
#
# temperature=0 makes the model's responses more
# deterministic and focused on the provided context.

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# Simple RAG

In [5]:
# ============================================================
# RAG Question Answering
# ============================================================

def rag_answer(
    query: str,
    top_k: int = 10
):
    """
    Generate an answer to a user query using a Retrieval-
    Augmented Generation (RAG) pipeline.

    The pipeline consists of three main stages:

        1. Retrieve relevant documents using hybrid search.
        2. Build a context from the retrieved passages.
        3. Provide the context and query to the LLM to
           generate the final answer.

    Args:
        query (str):
            The user's question.

        top_k (int):
            Number of relevant passages to retrieve from
            the vector database.

    Returns:
        str:
            The answer generated by the LLM.
    """

    # ========================================================
    # 1. Retrieve Relevant Documents
    # ========================================================
    # Use hybrid search to retrieve relevant passages.
    #
    # Hybrid search combines:
    #   - Dense semantic similarity
    #   - Sparse BM25 keyword matching
    #
    # This provides a stronger retrieval signal than relying
    # on only one retrieval method.
    # ========================================================

    results = hybrid_search(
        query=query,
        limit=top_k
    )


    # ========================================================
    # 2. Build Context
    # ========================================================
    # Extract the text from each retrieved passage and
    # combine them into a single context string.
    #
    # The context will be provided to the LLM as the only
    # source of information for answering the question.
    # ========================================================

    context = "\n\n".join(
        result.payload["text"]
        for result in results
    )


    # ========================================================
    # 3. Build LLM Prompt
    # ========================================================
    # Instruct the LLM to answer the question using only
    # the retrieved context.
    #
    # If the required information is not available in the
    # retrieved context, the model should explicitly state
    # that it does not know the answer.
    # ========================================================

    prompt = f"""
You are a helpful question-answering assistant.

Answer the user's question using only the provided context.

If the answer cannot be found in the context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{query}

Answer:
"""


    # ========================================================
    # 4. Generate Answer
    # ========================================================
    # Send the prompt to the LLM and extract the generated
    # response text.
    # ========================================================

    response = llm.invoke(prompt)


    # Return the final generated answer
    return response.content

In [6]:
rag_answer("what is Walgreens average salary ranges")

'The average Walgreens salary ranges from approximately $15,000 per year for Customer Service Associate / Cashier to $179,900 per year for District Manager.'

In [7]:
rag_answer("and how much store it opened")

"I don't know based on the provided context."

# RAG with Query Rewriter

In [8]:
# ============================================================
# Query Rewriting for Conversational RAG
# ============================================================

def rewrite_query(
    query: str,
    history
):
    """
    Rewrite the user's latest question into a standalone
    search query when it depends on previous conversation context.

    If the latest question is already standalone, the function
    returns it unchanged.

    The rewritten query is used for retrieval so that the
    retriever can understand references such as:
        - "he"
        - "that company"
        - "what about the second one?"
        - "how much does it cost?"

    Args:
        query (str):
            The user's latest question.

        history:
            Previous conversation messages. Each message is
            expected to contain `type` and `content` attributes.

    Returns:
        str:
            A standalone search query suitable for retrieval.
    """

    # ========================================================
    # 1. Format Conversation History
    # ========================================================
    # Convert the conversation history into a readable text
    # format that can be provided to the LLM.
    # ========================================================

    history_text = "\n".join(
        f"{msg.type}: {msg.content}"
        for msg in history
    )


    # ========================================================
    # 2. Build Query Rewriting Prompt
    # ========================================================
    # The LLM is instructed to determine whether the latest
    # question depends on previous conversation context.
    #
    # If it is already standalone, the original query must be
    # returned exactly as provided.
    #
    # If it depends on previous context, the LLM should rewrite
    # it into a complete standalone search query.
    # ========================================================

    prompt = f"""
Given the conversation history and the user's latest question,
decide whether the latest question depends on the conversation history.

Rules:

1. If the latest question is already standalone and can be understood
   without the conversation history, return it EXACTLY as provided.
   Do not rephrase it.

2. If the latest question depends on previous conversation context,
   rewrite it as a standalone search query.

3. The rewritten query must contain all important context from the
   conversation needed to understand the question.

4. Do not answer the question.

5. Return ONLY the final search query. Do not add explanations,
   labels, or quotation marks.

Conversation history:
{history_text}

Latest question:
{query}

Search query:
"""


    # ========================================================
    # 3. Generate Rewritten Query
    # ========================================================
    # Send the conversation history and latest question to
    # the LLM and retrieve the generated search query.
    # ========================================================

    response = llm.invoke(prompt)


    # ========================================================
    # 4. Return Clean Search Query
    # ========================================================
    # Remove leading/trailing whitespace before returning the
    # query to the retrieval pipeline.
    # ========================================================

    return response.content.strip()

In [9]:
# ============================================================
# Conversation History
# ============================================================
# Store previous user questions and assistant responses.
# This history is used by the query-rewriting step to resolve
# references and follow-up questions.
# ============================================================

chat_history = []


# ============================================================
# Conversational RAG Function
# ============================================================

def rag_chat(
    query: str,
    top_k: int = 10
):
    """
    Run a conversational Retrieval-Augmented Generation (RAG)
    pipeline that maintains conversation history.

    The pipeline consists of five steps:

        1. Rewrite the user's query using conversation history.
        2. Retrieve relevant documents using hybrid search.
        3. Build context from the retrieved passages.
        4. Generate an answer using the LLM.
        5. Add the current interaction to the conversation history.

    Args:
        query (str):
            The user's latest question.

        top_k (int):
            Number of relevant passages to retrieve.

    Returns:
        str:
            The answer generated by the LLM.
    """

    # ========================================================
    # 1. Rewrite Query Using Conversation History
    # ========================================================
    # The query-rewriting step determines whether the latest
    # question depends on previous messages.
    #
    # If necessary, it converts the question into a standalone
    # search query that can be understood by the retriever.
    # ========================================================

    rewritten_query = rewrite_query(
        query=query,
        history=chat_history
    )


    # Display the original and rewritten queries for debugging
    # and to understand how conversational queries are transformed.
    print("User query     :", query)
    print("Rewritten query:", rewritten_query)


    # ========================================================
    # 2. Retrieve Relevant Documents
    # ========================================================
    # Use the rewritten query with hybrid search.
    #
    # Hybrid search combines dense semantic retrieval with
    # sparse BM25 retrieval to find relevant passages.
    # ========================================================

    results = hybrid_search(
        query=rewritten_query,
        limit=top_k
    )


    # ========================================================
    # 3. Build Context
    # ========================================================
    # Extract the text from the retrieved passages and combine
    # them into a single context string for the LLM.
    # ========================================================

    context = "\n\n".join(
        result.payload["text"]
        for result in results
    )


    # ========================================================
    # 4. Generate Answer
    # ========================================================
    # Ask the LLM to answer the user's original question using
    # only the retrieved context.
    #
    # The conversation history is also provided so that the LLM
    # can understand the current question in its conversational
    # context.
    # ========================================================

    prompt = f"""
You are a helpful question-answering assistant.

Answer the user's latest question using only the provided context.

If the answer cannot be found in the context, say:
"I don't know based on the provided context."

Conversation history:
{chat_history}

Context:
{context}

Current question:
{query}

Answer:
"""


    # Generate the answer using the configured LLM
    response = llm.invoke(prompt)

    answer = response.content


    # ========================================================
    # 5. Update Conversation History
    # ========================================================
    # Store both the user's question and the assistant's answer.
    #
    # This allows future follow-up questions to use the previous
    # conversation as context.
    # ========================================================

    chat_history.append(
        HumanMessage(content=query)
    )

    chat_history.append(
        AIMessage(content=answer)
    )


    # ========================================================
    # 6. Return Answer
    # ========================================================

    return answer

In [10]:
rag_chat("what is Walgreens average salary ranges")

User query     : what is Walgreens average salary ranges
Rewritten query: Walgreens average salary ranges


'The average Walgreens salary ranges from approximately $15,000 per year for Customer Service Associate / Cashier to $179,900 per year for District Manager. Average Walgreens hourly pay ranges from approximately $7.35 per hour for Laboratory Technician to $68.90 per hour for Pharmacy Manager.'

In [12]:
rag_chat("and how much store it opened")

User query     : and how much store it opened
Rewritten query: how many stores has Walgreens opened


'Walgreens opened a total of 184 new locations in fiscal 2014.'